In [0]:
"""
06_work_order_events.py

Creates the Silver Work Order Events table.

Input:
    parsed_events

Output:
    work_order_events

Author:
Sumanth Vempalle

Version:
2.1.0
"""

from pyspark import pipelines as dp

from pyspark.sql.functions import col


# ============================================================
# Work Order Events
# ============================================================

@dp.table(
    name="work_order_events",
    comment="Validated manufacturing work order events.",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true",
    },
)

@dp.expect_or_drop(
    "valid_work_order_id",
    "work_order_id IS NOT NULL",
)

@dp.expect_or_drop(
    "valid_product_code",
    "product_code IS NOT NULL",
)

@dp.expect(
    "positive_quantity",
    "quantity > 0",
)

@dp.expect_or_drop(
    "valid_shift",
    "planned_shift IS NOT NULL",
)

def work_order_events():

    df = spark.readStream.table("parsed_events")

    return (

        df

        # -----------------------------------------
        # Keep only work order events
        # -----------------------------------------

        .filter(
            col("event_type") == "WORK_ORDER_CREATED"
        )

        # -----------------------------------------
        # Business columns
        # -----------------------------------------

        .select(

            "event_id",
            "event_timestamp",
            "event_version",

            "plant_code",

            "work_order_id",

            "product_code",

            "source_system",
            "correlation_id",

            "silver_processing_timestamp",

            "payload.sap_order_number",

            "payload.quantity",

            "payload.priority",

            "payload.planned_shift",

            "payload.routing_version",

            "payload.planner",

            "payload.status",

            "payload.product_name",

            "payload.family",

            "payload.rated_voltage_kv",

        )

    )